# Scube Mouse Hypothalamus (2/2): Alignment of consecutive ST slices

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
import SPACEL
from SPACEL import Scube

In [3]:
seed = 14
from SPACEL.setting import set_environ_seed
set_environ_seed(seed)

Setting environment seed: 14


## Spatial transcriptomic data import

The input data type of Scube is a list of anndata objects for each slice. The anndata objects of each slice must contain a column of spatial domain for alignment in ``.obs``.

In [ ]:
adata = sc.read(f'./{seed}/lymph_Splane.h5ad')
adata.obs['slice_id'] = adata.obs.index.astype(str).str.split('_').str[0]

adata[0].obs

,cell_ID_mask,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,n_reads,reads_per_counts,n_joined,exact_entropy,theoretical_entropy,...,Macrophages,Mast_cells,Plasma_IgA,Plasma_IgG,T_cell,Tumor,Tumor_Keratin_Pearl,unknown,spatial_domain,slice_id
2_307,307,246,285.0,32.0,11.22807,443,1.554386,296,1.9025,1.913041,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,10,2


In [5]:
slices = ['2', '3', '4', '5', '6', '7', '9', '11', '17', '18', '19', '23', '24', '25', '26', '28', '33', '34', '36']

adata_list=[]

for i in range(19):
    
    adata_list.append(adata[adata.obs['slice_id'] == slices[i]])
    adata_list[i].obs.spatial_domain = adata_list[i].obs.spatial_domain.astype('category')

In [6]:
adata_list[1]

AnnData object with n_obs × n_vars = 52405 × 28943
    obs: 'cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined', 'exact_entropy', 'theoretical_entropy', 'exact_compression', 'theoretical_compression', 'n_counts', 'annotation', 'annotation_key', 'n_section', 'original_clusters', 'CAF', 'CAM', 'Cortex_CCL21', 'Cytotoxic_IFN_signaling', 'Dendritic_cells', 'Fibroblasts', 'Germinal_Center_Plasma_IgM_B_cell', 'High_Endothelial_Venules', 'M1_macrophages', 'Macrophages', 'Mast_cells', 'Plasma_IgA', 'Plasma_IgG', 'T_cell', 'Tumor', 'Tumor_Keratin_Pearl', 'unknown', 'spatial_domain', 'slice_id'
    obsm: 'spatial', 'spatial_3d_aligned'
    layers: 'raw'

Plot all raw slices stacked in a single figure. Cells colored by spatial domain.First, we define the color of each spatial domain.

## Pairwise alignment

Here, we perform pairwise alignment of slices. The ``cluster_key`` is a column of ``.obs`` which contain spatial domain information used for alignment. The ``n_neighbors`` affect how much neighbors are considered when calculate similarity of two slices. The ``knn_exclude_cutoff`` affect max distance of neighbors. By default, ``n_neighbors = 15`` and ``knn_exclude_cutoff = n_neighbors`` means a spot/cell in source slices consider 15 neighbors in target slices, and neighbor will be exclude when the distance of neighbors larger than the median of neareast 30 neighbors distance in all spot/cell in target slice. ``p`` affect the exponent of overlap penalty, and a larger p means stronger partial alignment capability.

In [ ]:
Scube.align(adata_list,
      cluster_key='spatial_domain', 
      n_neighbors = 15, 
      n_threads=10,
      p=2,
      seed=seed,
     )

Start alignment...
Alignment slice 1 to 0
Alignment slice 2 to 1
Alignment slice 3 to 2
Alignment slice 4 to 3
Alignment slice 5 to 4
Alignment slice 6 to 5
Alignment slice 7 to 6
Alignment slice 8 to 7
Alignment slice 9 to 8
Alignment slice 10 to 9
Alignment slice 11 to 10
Alignment slice 12 to 11
Alignment slice 13 to 12
Alignment slice 14 to 13
Alignment slice 15 to 14
Alignment slice 16 to 15
Alignment slice 17 to 16
Alignment slice 18 to 17
Runtime: 13769.258447647095 s


After alignment, the new spots/cells location coordinates saved in ``.obsm['spatial_aligned']`` of anndata object of each slices. When ``write_loc_path`` provied by users, the location coordinates of all spots/cells of all slices will be saved in ``write_loc_path``.

In [8]:
adata_list[0].obs['original_clusters'].unique()

['Tumor', 'unknown', 'Plasma_IgG', 'Fibroblasts', 'CAF', ..., 'High_Endothelial_Venules', 'Tumor_Keratin_Pearl', 'CAM', 'Macrophages', 'Cortex_CCL21']
Length: 15
Categories (15, object): ['CAF', 'CAM', 'Cortex_CCL21', 'Cytotoxic_IFN_signaling', ..., 'T_cell', 'Tumor', 'Tumor_Keratin_Pearl', 'unknown']

## LAA & ARI

In [9]:
from sklearn.metrics import pairwise
from sklearn.metrics import adjusted_rand_score

def cal_layer_based_alignment_result_mHypothalamus(alignment, s1, s2):
    #fei added
    labels = []
    labels.extend(s1.obs['original_clusters'])
    labels.extend(s2.obs['original_clusters'])

    res = []
    #l_dict = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6}
    cnt0 = 0
    ir = 0
    max_smallerThan0 = []
    sum_smallerThan0 = []
    min_smallerThan0 = []

    for i, elem in enumerate(alignment):
        #if np.sum(elem)>0:
        if np.max(elem) - np.min(elem) != 0:
            if labels[i] == labels[elem.argmax() + alignment.shape[0]]:
                cnt0 += 1
        else:
            ir += 1
            max_smallerThan0.append(np.max(elem))
            sum_smallerThan0.append(np.sum(elem))
            min_smallerThan0.append(np.min(elem))
    print("the number of sum alignment equals 0:", str(ir))
    if len(sum_smallerThan0)>0:
        print("sum in sum alignment equals 0 is between", np.min(sum_smallerThan0), np.max(sum_smallerThan0))
        print("max in sum alignment equals 0 is between", np.min(max_smallerThan0), np.max(max_smallerThan0))
        print("min in sum alignment equals 0 is between", np.min(min_smallerThan0), np.max(min_smallerThan0))

    res.extend([cnt0])
    return res


def compute_alignment_ari(pi, sliceA, sliceB):
    mapped_clusters = []
    for j in range(pi.shape[1]):
        mapping = pi[:, j]
        #original: if np.sum(mapping) > 0:
        if np.max(mapping) - np.min(mapping) != 0:
            i = np.argmax(mapping)
            mapped_clusters.append(sliceA.obs['original_clusters'][i])
        else:
            mapped_clusters.append("NULL")
    assert len(sliceB.obs['original_clusters']) == len(mapped_clusters)
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []
    for i in range(len(sliceB.obs['original_clusters'])):
        if mapped_clusters[i] != "NULL":
            true_clusters_mapped_region.append(sliceB.obs['original_clusters'][i])
            mapped_clusters_mapped_region.append(mapped_clusters[i])

    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
    return ari

In [ ]:
slices = ['2', '3', '4', '5', '6', '7', '9', '11', '17', '18', '19', '23', '24', '25', '26', '28', '33', '34', '36']

ari_values = []
laa_values = []

for i in range(1, len(slices)):
    target_ind = i-1
    source_ind = i
    f1 = adata_list[target_ind].obsm['spatial_aligned'].to_numpy()
    f2 = adata_list[source_ind].obsm['spatial_aligned'].to_numpy()

    alignment = pairwise.euclidean_distances(f1, f2)
    alignment = np.exp(-alignment)
    
    name1 = slices[target_ind].split('.')[-1]
    name2 = slices[source_ind].split('.')[-1]
    # Save as .pkl
#     alignment_df = pd.DataFrame(alignment)
#     alignment_df.to_pickle(f'./lymph/{seed}/{name1}_{name2}_AlignmentPi.pkl')
    
    res = cal_layer_based_alignment_result_mHypothalamus(alignment, adata_list[source_ind], adata_list[target_ind])
    laa = res[0]/len(adata_list[source_ind].obs['original_clusters'])
    ari = compute_alignment_ari(alignment, adata_list[target_ind], adata_list[source_ind])
    ari_values.append(ari)
    laa_values.append(laa)
    print(laa, ari)

# Create DataFrame with desired format
header_row = [f"{slices[i-1]}_{slices[i]}" for i in range(1, len(slices))]
ari_df = pd.DataFrame([ari_values], columns=header_row)
laa_df = pd.DataFrame([laa_values], columns=header_row)

the number of sum alignment equals 0: 1079
sum in sum alignment equals 0 is between 0.0 0.0
max in sum alignment equals 0 is between 0.0 0.0
min in sum alignment equals 0 is between 0.0 0.0
0.21038068886556627 0.3858481745163953
the number of sum alignment equals 0: 9924
sum in sum alignment equals 0 is between 0.0 0.0
max in sum alignment equals 0 is between 0.0 0.0
min in sum alignment equals 0 is between 0.0 0.0
0.047041146555709665 0.19855328386149065
the number of sum alignment equals 0: 11286
sum in sum alignment equals 0 is between 0.0 0.0
max in sum alignment equals 0 is between 0.0 0.0
min in sum alignment equals 0 is between 0.0 0.0
0.12497617383569477 0.26736408485091306
the number of sum alignment equals 0: 20034
sum in sum alignment equals 0 is between 0.0 0.0
max in sum alignment equals 0 is between 0.0 0.0
min in sum alignment equals 0 is between 0.0 0.0
0.09176501102321359 0.20154590224071908
the number of sum alignment equals 0: 4722
sum in sum alignment equals 0 is be

In [ ]:
# Save as a CSV file
ari_df.to_csv(f'./{seed}/lymph_ARI.csv')
laa_df.to_csv(f'./{seed}/lymph_LAA.csv')

## Plot alignment results

Save all spots/cells aligned coordiantes and colors in a DataFrame.

In [12]:
c_dict = {
    'CAF': '#1f77b4', 
    'CAM': '#aec7e8', 
    'Cortex_CCL21': '#ff7f0e', 
    'Cytotoxic_IFN_signaling': '#ffbb78', 
    'Dendritic_cells': '#2ca02c', 
    'Fibroblasts': '#98df8a', 
    'Germinal_Center_Plasma_IgM_B_cell': '#d62728', 
    'High_Endothelial_Venules': '#ff9896', 
    'M1_macrophages': '#9467bd', 
    'Macrophages': '#c5b0d5', 
    'Mast_cells': '#8c564b', 
    'Plasma_IgA': '#c49c94', 
    'Plasma_IgG': '#e377c2', 
    'T_cell': '#f7b6d2', 
    'Tumor': '#7f7f7f', 
    'Tumor_Keratin_Pearl': '#c7c7c7', 
    'unknown': '#bcbd22'
    }

In [ ]:
coo = pd.DataFrame()

for i in range(len(adata_list)):
    loc = adata_list[i].obsm['spatial_aligned'].copy()
    loc['Z'] = i
    loc['slice'] = slices[i].split('.')[-1]
    loc['original_clusters'] = adata_list[i].obs['original_clusters']
    loc['celltype_colors'] = adata_list[i].obs['original_clusters'].replace(c_dict)
    coo = pd.concat([coo,loc],axis=0)
    
coo.to_csv(f'./{seed}/SPACEL_lymph_aligned_coordinates_colors.csv')

In [14]:
coo

,X,Y,Z,slice,original_clusters,celltype_colors
2_307,-10010.013672,-197.266602,0,2,Tumor,#7f7f7f
2_309,-10010.239258,402.979004,0,2,unknown,#bcbd22
2_446,-9956.068359,594.534180,0,2,Tumor,#7f7f7f
2_655,-9776.084961,1179.823730,0,2,Tumor,#7f7f7f
2_656,-9774.104492,1216.675293,0,2,Tumor,#7f7f7f
...,...,...,...,...,...,...
36_90890,-1876.795410,10247.372070,18,36,M1_macrophages,#9467bd
36_90900,-2016.920898,10258.195312,18,36,Germinal_Center_Plasma_IgM_B_cell,#d62728
36_90933,-1893.050537,10277.492188,18,36,M1_macrophages,#9467bd
36_90945,-2042.821289,10280.675781,18,36,M1_macrophages,#9467bd


Plot all aligned slices stacked in 3D constructed by Scube

In [ ]:
def plot_3d(
    loc,
    val=None,
    color=None,
    figsize=(12,12),
    return_fig=True,
    elev=None,
    azim=None,
    roll=None,
    xlim=None,
    ylim=None,
    zlim=None,
    frameon=True,
    save_path=None,
    save_dpi=150,
    show=True,
    *args,
    **kwargs
):
    """Plot all slices stacked in 3D

    Plot all slices stacked with a given number of subplots rows and columns. Spots/cells colored by spatial domain.

    Args:
        loc: An array of the coordinates of each spots/cells in all slices, which the first three columns are X-axis, Y-axis, Z-axis coordinates.
        val: The colors of each spots/cells given in loc use to plot.
        color: The colors of each spots/cells given in loc use to plot.
        figsize: Size of the figure.
        return_fig: Whether to return the figure.
        elev: The elevation angle in the vertical plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        azim: The azimuth angle in the horizontal plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        xlim: A tuple given the left and right xlims in X-axis.
        ylim: A tuple given the left and right xlims in Y-axis.
        zlim: A tuple given the left and right xlims in Z-axis.
        frameon: Whether to hide the coordinate axes.
        save_path: A string representing the path directory where the figure saved.
        save_dpi: The resolution in dots per inch.
        show: Whether to show the figure.

    Returns:
        A ``matplotlib.figure.Figure`` object
    """
    if 'marker' not in kwargs.keys():
        kwargs['marker'] = 'o'
    if 's' not in kwargs.keys():
        kwargs['s'] = 5
    if 'cmap' not in kwargs.keys():
        kwargs['cmap'] = 'Spectral_r'
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    if (elev is not None) and (azim is not None) and (roll is not None):
        ax.view_init(elev, azim, roll)
    if color is None:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=val,*args,**kwargs)
    else:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=color,*args,**kwargs)
    
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    if not frameon:
        plt.axis('off')
        plt.grid(b=None)
    if save_path is not None:
        print(save_path)
        plt.savefig(save_path,dpi=save_dpi,bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    if return_fig:
        return fig

In [ ]:
seed = 2024
df = pd.read_csv(f'./{seed}/SPACEL_lymph_aligned_coordinates_colors.csv')
coords = df[['X', 'Y', 'Z']]
# coords['Z'] = coords['Z']

In [ ]:
fig = plot_3d(coords.values, val=None, color=df['celltype_colors'], elev=30, azim=30, roll=0, frameon=False, show=True, s=1, save_dpi=300, return_fig=True) # Scube is from SPACEL

fig.savefig(f'./{seed}/SPACEL_gtAndScube_lymph_3dviz_angle30_30_s=1.pdf', bbox_inches="tight")